# ML Results Quick Review

Быстрый просмотр результатов `run_ml_baselines.py` без переобучения: метрики, прогнозы, trials, best params, важность признаков и сравнение по широтным зонам.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

BASE = Path('..') if Path('../run_ml_baselines.py').exists() else Path('.')
RUN_DIR = BASE / 'artifacts' / 'baseline_v0.1'
STATION = None      # пример: 'TR169'; None = первая доступная
HORIZON = None      # пример: '1h'; None = все
MODEL = None        # пример: 'Optuna_CatBoost'; None = все
TOP_N = 20

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
print('RUN_DIR =', RUN_DIR.resolve())


In [ ]:
def read_table(path):
    path = Path(path)
    if path.with_suffix('.parquet').exists():
        return pd.read_parquet(path.with_suffix('.parquet'))
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()

metrics = read_table(RUN_DIR / 'metrics.csv')
summary = read_table(RUN_DIR / 'metrics_summary.csv')
registry = read_table(RUN_DIR / 'model_registry.csv')
best_params = read_table(RUN_DIR / 'best_params.csv')
trials = read_table(RUN_DIR / 'optuna_trials.csv')
importance = read_table(RUN_DIR / 'feature_importance.csv')
station_summary = read_table(RUN_DIR / 'station_run_summary.csv')
manifest_path = RUN_DIR / 'run_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}

print('metrics:', metrics.shape)
print('summary:', summary.shape)
print('registry:', registry.shape)
print('best_params:', best_params.shape)
print('trials:', trials.shape)
print('importance:', importance.shape)
print('station_summary:', station_summary.shape)
display(station_summary.head(20))


## Фильтры

In [ ]:
def apply_filters(df):
    if df.empty:
        return df
    out = df.copy()
    if STATION and 'station' in out.columns:
        out = out[out['station'] == STATION]
    if HORIZON and 'horizon' in out.columns:
        out = out[out['horizon'] == HORIZON]
    if MODEL and 'model' in out.columns:
        out = out[out['model'] == MODEL]
    return out

m = apply_filters(metrics)
r = apply_filters(registry)
bp = apply_filters(best_params)
tr = apply_filters(trials)
fi = apply_filters(importance)
print('filtered metrics:', m.shape)
display(m.head())


## Лучшие модели по RMSE

In [ ]:
if not m.empty:
    best = (
        m.sort_values('rmse')
        .groupby(['station', 'horizon', 'train_days'], dropna=False)
        .head(1)
        .sort_values(['station', 'horizon', 'train_days'])
    )
    cols = [c for c in ['station','latitude_zone','horizon','train_days','split_id','model','n','mae','rmse','r2','corr','bias','best_val_score'] if c in best.columns]
    display(best[cols].head(TOP_N))
else:
    print('Нет metrics. Сначала запусти run_ml_baselines.py.')


In [ ]:
if not m.empty:
    metric_cols = [c for c in ['rmse','mae','r2','corr','bias'] if c in m.columns]
    agg = m.groupby(['horizon','model'], dropna=False)[metric_cols].mean(numeric_only=True).reset_index()
    fig = px.bar(agg, x='model', y='rmse', color='horizon', barmode='group', title='Mean RMSE by model and horizon', height=520)
    fig.update_xaxes(tickangle=35)
    fig.show()


## Широтные зоны

In [ ]:
if not m.empty and 'latitude_zone' in m.columns:
    metric_cols = [c for c in ['rmse','mae','r2','corr','bias'] if c in m.columns]
    zone = m.groupby(['latitude_zone','horizon','model'], dropna=False)[metric_cols].mean(numeric_only=True).reset_index()
    display(zone.sort_values(['latitude_zone','horizon','rmse']).head(TOP_N))
    fig = px.bar(zone, x='model', y='rmse', color='latitude_zone', facet_col='horizon', barmode='group', title='RMSE by latitude zone', height=520)
    fig.update_xaxes(tickangle=35)
    fig.show()
else:
    print('latitude_zone пока нет в metrics или metrics пустой')


## Прогнозы станции

In [ ]:
def load_station_predictions(station=None):
    stations_dir = RUN_DIR / 'stations'
    if not stations_dir.exists():
        return pd.DataFrame()
    if station is None:
        station_dirs = sorted(p for p in stations_dir.iterdir() if p.is_dir())
        if not station_dirs:
            return pd.DataFrame()
        station_dir = station_dirs[0]
    else:
        station_dir = stations_dir / station
    parquet = station_dir / 'predictions.parquet'
    csv = station_dir / 'predictions.csv'
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        return pd.read_csv(csv, parse_dates=['time_utc','target_time_utc'])
    return pd.DataFrame()

pred = load_station_predictions(STATION)
if not pred.empty:
    pred['time_utc'] = pd.to_datetime(pred['time_utc'], utc=True, errors='coerce')
    pred = apply_filters(pred)
print('predictions:', pred.shape)
display(pred.head())


In [ ]:
if not pred.empty:
    plot_station = pred['station'].iloc[0]
    plot_horizon = HORIZON or pred['horizon'].iloc[0]
    plot_model = MODEL or pred['model'].iloc[0]
    subset = pred[(pred['station'] == plot_station) & (pred['horizon'] == plot_horizon) & (pred['model'] == plot_model)].copy()
    subset = subset.sort_values('time_utc').head(96 * 14)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=subset['time_utc'], y=subset['actual'], mode='lines', name='actual'))
    fig.add_trace(go.Scatter(x=subset['time_utc'], y=subset['predicted'], mode='lines', name='predicted'))
    fig.update_layout(title=f'{plot_station} {plot_horizon} {plot_model}: actual vs predicted', xaxis_title='Time UTC', yaxis_title='foF2 MHz', height=520)
    fig.show()
else:
    print('Нет predictions для просмотра')


## Best Params ? Trials

In [ ]:
if not bp.empty:
    cols = [c for c in ['station','latitude_zone','horizon','train_days','split_id','model','search_method','best_value','best_params'] if c in bp.columns]
    bp_view = bp.loc[:, cols].copy()
    sort_cols = [c for c in ['station','horizon','best_value'] if c in bp_view.columns]
    if sort_cols:
        bp_view = bp_view.sort_values(by=sort_cols)
    display(bp_view.head(TOP_N))
else:
    print('best_params пустой')

if not tr.empty:
    cols = [c for c in ['station','latitude_zone','horizon','train_days','split_id','model','trial_number','value','state','params'] if c in tr.columns]
    tr_view = tr.loc[:, cols].copy()
    if 'value' in tr_view.columns:
        tr_view = tr_view.sort_values(by='value')
    display(tr_view.head(TOP_N))
else:
    print('trials пустой')


## Важность признаков

In [ ]:
if not fi.empty:
    top_features = (
        fi.groupby(['model','feature'], dropna=False)['importance_abs']
        .mean()
        .reset_index()
        .sort_values('importance_abs', ascending=False)
        .head(TOP_N)
    )
    display(top_features)
    fig = px.bar(top_features.sort_values('importance_abs'), x='importance_abs', y='feature', color='model', orientation='h', title='Top feature importance / coefficients', height=650)
    fig.show()
else:
    print('feature_importance пустой')


## Manifest

In [ ]:
if manifest:
    print(json.dumps({k: manifest.get(k) for k in ['stations','horizons','metrics_rows','prediction_rows','trial_rows','best_param_rows','feature_importance_rows','result_layout']}, ensure_ascii=False, indent=2))
else:
    print('run_manifest.json пока нет')
